# 도구 선택(tool_choice)

도구 사용에는 Claude가 도구를 어떻게 호출할지 지정할 수 있는 `tool_choice` 파라미터가 있습니다. 이 노트북에서는 이것이 어떻게 동작하고 언제 사용하는지 살펴봅니다. 더 진행하기 전에 Claude 도구 사용의 기본기를 익혀 두시기 바랍니다.

`tool_choice` 파라미터에는 세 가지 선택지가 있습니다.

* `auto` — 제공된 도구를 호출할지 말지를 Claude가 스스로 결정하게 합니다
* `tool` — 특정 도구를 항상 사용하도록 Claude에 강제합니다
* `any` — 제공된 도구 중 하나는 반드시 사용해야 한다고 알리되, 특정 도구를 지정하지는 않습니다

각 선택지를 자세히 살펴보겠습니다. 먼저 Anthropic SDK를 가져옵니다:

In [31]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## Auto

`tool_choice`를 `auto`로 설정하면 도구를 사용할지 말지를 모델이 자동으로 결정합니다. 도구를 사용할 때의 기본 동작입니다.

이를 보여 주기 위해 Claude에 가짜 웹 검색 도구를 제공하겠습니다. Claude에 여러 질문을 던질 텐데, 일부는 웹 검색 도구 호출이 필요하고 나머지는 Claude가 스스로 답할 수 있는 것들입니다.

먼저 `web_search`라는 도구를 정의하겠습니다. 데모를 단순하게 유지하려고 실제로 웹을 검색하지는 않는다는 점에 유의하세요:

In [137]:
def web_search(topic):
    print(f"pretending to search the web for {topic}")


web_search_tool = {
    "name": "web_search",
    "description": "A tool to retrieve up to date information on a given topic by searching the web",
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "The topic to search the web for"},
        },
        "required": ["topic"],
    },
}

다음으로 user_query를 받아 `web_search_tool`과 함께 Claude에 전달하는 함수를 작성합니다.

그리고 `tool_choice`를 `auto`로 설정합니다:

```py
tool_choice={"type": "auto"}
```

전체 함수는 다음과 같습니다:

In [145]:
from datetime import date


def chat_with_web_search(user_query):
    messages = [{"role": "user", "content": user_query}]

    system_prompt = f"""
    Answer as many questions as you can using your existing knowledge.
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
    """

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=[web_search_tool],
    )
    last_content_block = response.content[-1]
    if last_content_block.type == "text":
        print("Claude did NOT call a tool")
        print(f"Assistant: {last_content_block.text}")
    elif last_content_block.type == "tool_use":
        print("Claude wants to use a tool")
        print(last_content_block)

도구를 쓰지 않고도 Claude가 답할 수 있는 질문부터 시작해 보겠습니다:

In [139]:
chat_with_web_search("What color is the sky?")

Claude did NOT call a tool
Assistant: The sky appears blue during the day. This is because the Earth's atmosphere scatters more blue light from the sun than other colors, making the sky look blue.


"하늘은 무슨 색인가요?"라고 물으면 Claude는 도구를 사용하지 않습니다. 이번에는 답하려면 웹 검색 도구가 필요한 질문을 던져 보겠습니다:

In [140]:
chat_with_web_search("Who won the 2024 Miami Grand Prix?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_018nwaaRebX33pHqoZZXDaSw', input={'topic': '2024 Miami Grand Prix winner'}, name='web_search', type='tool_use')


"2024년 마이애미 그랑프리 우승자는 누구인가요?"라고 묻자 Claude가 웹 검색 도구를 사용했습니다!

예시를 몇 개 더 살펴보겠습니다:

In [141]:
# Claude should NOT need to use the tool for this:
chat_with_web_search("Who won the superbowl in 2022?")

Claude did NOT call a tool
Assistant: The Los Angeles Rams won Super Bowl LVI in 2022, defeating the Cincinnati Bengals by a score of 23-20. The game was played on February 13, 2022 at SoFi Stadium in Inglewood, California.


In [144]:
# Claude SHOULD use the tool for this:
chat_with_web_search("Who won the superbowl in 2024?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_016XPwcprHAgYJBtN7A3jLhb', input={'topic': '2024 Super Bowl winner'}, name='web_search', type='tool_use')


### 프롬프트가 중요합니다!

`tool_choice`를 `auto`로 쓸 때는 상세한 프롬프트를 작성하는 데 시간을 들이는 것이 중요합니다. Claude가 도구를 지나치게 열심히 호출하려는 경우가 종종 있습니다. 상세한 프롬프트는 언제 도구를 호출하고 언제 호출하지 말아야 할지 Claude가 판단하는 데 도움이 됩니다. 위 예제에서는 시스템 프롬프트에 구체적인 지시를 포함했습니다:

```py
 system_prompt=f"""
    Answer as many questions as you can using your existing knowledge.  
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
"""
```

## 특정 도구 강제하기

`tool_choice`로 특정 도구를 사용하도록 Claude에 강제할 수 있습니다. 아래 예제에서는 간단한 도구 두 개를 정의했습니다.
* `print_sentiment_scores` — 감성 분석 데이터를 담은 잘 구조화된 JSON을 출력하도록 Claude를 "유도하는" 도구입니다. 이 접근법에 대한 자세한 내용은 [Claude와 도구 사용으로 구조화된 JSON 추출하기](https://github.com/anthropics/anthropic-cookbook/blob/main/tool_use/extracting_structured_json.ipynb)를 참고하세요
* `calculator` — 두 숫자를 받아 더하는 아주 간단한 계산기 도구입니다

In [111]:
tools = [
    {
        "name": "print_sentiment_scores",
        "description": "Prints the sentiment scores of a given tweet or piece of text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "positive_score": {
                    "type": "number",
                    "description": "The positive sentiment score, ranging from 0.0 to 1.0.",
                },
                "negative_score": {
                    "type": "number",
                    "description": "The negative sentiment score, ranging from 0.0 to 1.0.",
                },
                "neutral_score": {
                    "type": "number",
                    "description": "The neutral sentiment score, ranging from 0.0 to 1.0.",
                },
            },
            "required": ["positive_score", "negative_score", "neutral_score"],
        },
    },
    {
        "name": "calculator",
        "description": "Adds two number",
        "input_schema": {
            "type": "object",
            "properties": {
                "num1": {"type": "number", "description": "first number to add"},
                "num2": {"type": "number", "description": "second number to add"},
            },
            "required": ["num1", "num2"],
        },
    },
]

목표는 트윗을 받아 기본적인 감성 분석 결과를 출력하는 `analyze_tweet_sentiment` 함수를 작성하는 것입니다. 최종적으로는 감성 분석 도구를 사용하도록 Claude에 "강제"할 것이지만, 먼저 도구 사용을 강제하지 **않으면** 어떤 일이 벌어지는지 보여 드리겠습니다.

이 첫 번째 "나쁜" 버전의 `analyze_tweet_sentiment` 함수에서는 Claude에 두 도구를 모두 제공합니다. 비교를 위해 우선 tool_choice를 "auto"로 설정합니다:

```py
tool_choice={"type": "auto"}
```

특정 도구 사용을 강제했을 때의 효과를 더 잘 드러내기 위해, 일부러 잘 작성된 프롬프트를 제공하지 않았다는 점에 유의하세요.

In [124]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": query}],
    )
    print(response)

"Holy cow, I just made the most incredible meal!"라는 트윗으로 함수를 호출하면 어떻게 되는지 살펴보겠습니다.

In [125]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_01ApgXx7W7qsDugdaRWh6p21', content=[TextBlock(text="That's great to hear! I don't actually have the capability to assess sentiment from text, but it sounds like you're really excited and proud of the incredible meal you made. Cooking something delicious that you're proud of can definitely give a sense of accomplishment and happiness. Well done on creating such an amazing dish!", type='text')], model='claude-sonnet-4-6', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=429, output_tokens=69))


Claude가 감성 분석 도구를 호출하지 않습니다:
> "That's great to hear! I don't actually have the capability to assess sentiment from text, but it sounds like you're really excited and proud of the incredible meal you made

이번에는 누군가 이렇게 트윗했다고 해 봅시다: "I love my cats! I had four and just adopted 2 more! Guess how many I have now?"

In [128]:
analyze_tweet_sentiment(
    "I love my cats! I had four and just adopted 2 more! Guess how many I have now?"
)

ToolsBetaMessage(id='msg_staging_018gTrwrx6YwBR2jjhdPooVg', content=[TextBlock(text="That's wonderful that you love your cats and adopted two more! To figure out how many cats you have now, I can use the calculator tool:", type='text'), ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=442, output_tokens=101))


Claude가 계산기 도구를 호출하려고 합니다:

> ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')

지금 구현은 우리가 원하는 대로 동작하지 않는 것이 분명합니다(대체로 실패하도록 일부러 구성했기 때문입니다).

이제 `tool_choice`를 수정해 `print_sentiment_scores` 도구를 **항상** 사용하도록 Claude에 강제해 보겠습니다:

```py
tool_choice={"type": "tool", "name": "print_sentiment_scores"}
```

`type`을 `tool`로 설정하는 것에 더해, 특정 도구 이름도 함께 지정해야 합니다.

In [132]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "tool", "name": "print_sentiment_scores"},
        messages=[{"role": "user", "content": query}],
    )
    print(response)

이제 앞에서와 같은 프롬프트를 던지면 Claude는 항상 `print_sentiment_scores` 도구를 호출합니다:

In [133]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_018GtYk8Xvee3w8Eeh6pbgoq', content=[ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=527, output_tokens=79))


Claude가 `print_sentiment_scores` 도구를 호출합니다:

> ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')

"수학 같은" 트윗으로 Claude를 헷갈리게 하려 해도, 여전히 항상 `print_sentiment_scores` 도구를 호출합니다:

In [134]:
analyze_tweet_sentiment(
    "I love my cats! I had four and just adopted 2 more! Guess how many I have now?"
)

ToolsBetaMessage(id='msg_staging_01RACamfrHdpvLxWaNwDfZEF', content=[ToolUseBlock(id='toolu_staging_01Wb6ZKSwKvqVSKLDAte9cKU', input={'positive_score': 0.8, 'negative_score': 0.0, 'neutral_score': 0.2}, name='print_sentiment_scores', type='tool_use')], model='claude-sonnet-4-6', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=540, output_tokens=79))


`print_sentiment_scores` 도구를 호출하도록 Claude에 강제하더라도, 기본적인 프롬프트 엔지니어링은 여전히 적용해야 합니다:

In [135]:
def analyze_tweet_sentiment(query):
    prompt = f"""
    Analyze the sentiment in the following tweet:
    <tweet>{query}</tweet>
    """

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": prompt}],
    )
    print(response)

## Any

`tool_choice`의 마지막 선택지는 `any`로, Claude에 "도구는 반드시 호출해야 하지만 어떤 것을 쓸지는 네가 골라라"라고 말하는 것입니다. Claude로 SMS 챗봇을 만든다고 생각해 보세요. 이 챗봇이 사용자와 실제로 "소통"할 수 있는 유일한 수단은 SMS 문자 메시지입니다.

아래 예제에서는 두 가지 도구를 사용할 수 있는 아주 간단한 문자 메시지 어시스턴트를 만듭니다.
* `send_text_to_user` — 사용자에게 문자 메시지를 보냅니다
* `get_customer_info` — 사용자 이름으로 고객 데이터를 조회합니다

핵심은 이 두 도구 중 하나를 항상 호출하고, 도구를 쓰지 않는 응답은 절대 하지 않는 챗봇을 만드는 것입니다. 어떤 상황에서든 Claude는 문자 메시지를 보내려고 시도하거나, 고객 정보를 더 얻기 위해 `get_customer_info`를 호출해야 합니다.

무엇보다 중요한 것은 `tool_choice`를 "any"로 설정하는 것입니다:

```py
tool_choice={"type": "any"}
```

In [162]:
def send_text_to_user(text):
    # Sends a text to the user
    # We'll just print out the text to keep things simple:
    print(f"TEXT MESSAGE SENT: {text}")


def get_customer_info(username):
    return {
        "username": username,
        "email": f"{username}@email.com",
        "purchases": [
            {"id": 1, "product": "computer mouse"},
            {"id": 2, "product": "screen protector"},
            {"id": 3, "product": "usb charging cable"},
        ],
    }


tools = [
    {
        "name": "send_text_to_user",
        "description": "Sends a text message to a user",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The piece of text to be sent to the user via text message",
                },
            },
            "required": ["text"],
        },
    },
    {
        "name": "get_customer_info",
        "description": "gets information on a customer based on the customer's username.  Response includes email, username, and previous purchases. Only call this tool once a user has provided you with their username",
        "input_schema": {
            "type": "object",
            "properties": {
                "username": {
                    "type": "string",
                    "description": "The username of the user in question. ",
                },
            },
            "required": ["username"],
        },
    },
]

system_prompt = """
All your communication with a user is done via text message.
Only call tools when you have enough information to accurately call them.
Do not call the get_customer_info tool until a user has provided you with their username. This is important.
If you do not know a user's username, simply ask a user for their username.
"""


def sms_chatbot(user_message):
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "any"},
        messages=messages,
    )
    if response.stop_reason == "tool_use":
        last_content_block = response.content[-1]
        if last_content_block.type == "tool_use":
            tool_name = last_content_block.name
            tool_inputs = last_content_block.input
            print(f"=======Claude Wants To Call The {tool_name} Tool=======")
            if tool_name == "send_text_to_user":
                send_text_to_user(tool_inputs["text"])
            elif tool_name == "get_customer_info":
                print(get_customer_info(tool_inputs["username"]))
            else:
                print("Oh dear, that tool doesn't exist!")

    else:
        print("No tool was called. This shouldn't happen!")

간단한 것부터 시작해 보겠습니다:

In [163]:
sms_chatbot("Hey there! How are you?")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hello! I'm doing well, thanks for asking. How can I assist you today?


Claude가 `send_text_to_user` 도구를 호출해 응답합니다.

다음으로 조금 더 까다로운 것을 물어보겠습니다:

In [164]:
sms_chatbot("I need help looking up an order")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hi there, to look up your order details I'll need your username first. Can you please provide me with your username?


Claude가 사용자에게 사용자 이름을 알려 달라는 문자 메시지를 보내려고 합니다.

이제 사용자 이름을 알려 주면 어떻게 되는지 살펴보겠습니다:

In [165]:
sms_chatbot("I need help looking up an order.  My username is jenny76")

=======Claude Wants To Call The get_customer_info Tool=======
{'username': 'jenny76', 'email': 'jenny76@email.com', 'purchases': [{'id': 1, 'product': 'computer mouse'}, {'id': 2, 'product': 'screen protector'}, {'id': 3, 'product': 'usb charging cable'}]}


기대한 대로 Claude가 `get_customer_info` 도구를 호출했습니다!

Claude에 아무 의미 없는 메시지를 보내더라도, 여전히 두 도구 중 하나를 호출합니다:

In [166]:
sms_chatbot("askdj aksjdh asjkdbhas kjdhas 1+1 ajsdh")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: I'm afraid I didn't understand your query. Could you please rephrase what you need help with?
